In [ ]:
# This file is part of the article:
# "Leveraging Remote Traffic Data for Local Air Pollutant Estimation:
# A Scenario-Based Machine Learning Study Across London Monitoring Sites"
#
# Copyright (C) 2026 The authors
#
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License
# along with this program. If not, see <https://www.gnu.org/licenses/>.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any
import os
import lightgbm as lgb
from xgboost import XGBRegressor
import joblib
from pathlib import Path
from sklearn.metrics.pairwise import haversine_distances
import json
import joblib
import lightgbm as lgb
from xgboost import XGBRegressor
import shap
from utils import *

# Table 1

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import haversine_distances
import pandas as pd
import os
df = pd.read_csv("all_stations_more.csv")
df["Site Name"] = (
    df["Site Name"]
    .str.replace("\xa0", "", regex=False)
    .str.replace("Â", "", regex=False)
    .str.strip()
)
df["Sitename_type"] = df["Site Name"] + " " + df["Environment Type"]
# change to radians
coords = np.radians(df[["Latitude", "Longitude"]])

# Distance matrixs (in radians)
dist_matrix = haversine_distances(coords)

# Changing to km
earth_radius_km = 6371
dist_matrix_km = dist_matrix * earth_radius_km
dist_matrix_km = np.round(dist_matrix_km, 2)
dist_df = pd.DataFrame(
    dist_matrix_km,
    index=df["Sitename_type"],
    columns=df["Sitename_type"]
)

dist_df.to_csv("distance_matrix_km_btw_stations.csv", index=True)
stations = ['Camden Kerbside Urban Traffic','London Marylebone Road Urban Traffic','Westminster - Oxford Street Urban Traffic', 'Wandsworth - Putney High Street Urban Traffic', 'Camden - Euston Road Urban Traffic']

def find_nearest_by_type(dist_df, stations, suffix, k=2):
    """
    Find k nearest stations ending with a given suffix
    for each station in stations list.
    """
    results = {}
    candidates = [s for s in dist_df.index if s.endswith(suffix) and s not in stations]    
    for station in stations:        
        valid_candidates = [c for c in candidates if c != station] 
        distances = dist_df.loc[station, valid_candidates]   
        nearest = distances.sort_values().head(k)        
        results[station] = nearest    
    return results

nearest_background = find_nearest_by_type(
    dist_df,
    stations,
    suffix="Background",
    k=2
)

nearest_traffic = find_nearest_by_type(
    dist_df,
    stations,
    suffix="Traffic",
    k=2
)

table_rows = {
    "1st nearest background station": {},
    "2nd nearest background station": {},
    "1st nearest traffic station": {},
    "2nd nearest traffic station": {}
}

for target_station, neighbours in nearest_background.items():
    for rank, (neighbour_station, distance_km) in enumerate(neighbours.items(), start=1):
        row_name = f"{rank}{'st' if rank == 1 else 'nd'} nearest background station"
        table_rows[row_name][target_station] = f"{neighbour_station} ({distance_km:.2f} km)"

for target_station, neighbours in nearest_traffic.items():
    for rank, (neighbour_station, distance_km) in enumerate(neighbours.items(), start=1):
        row_name = f"{rank}{'st' if rank == 1 else 'nd'} nearest traffic station"
        table_rows[row_name][target_station] = f"{neighbour_station} ({distance_km:.2f} km)"

df_table = pd.DataFrame.from_dict(table_rows, orient="index")

df_table.index.name = "Distance between stations"

df_table.to_csv("Table1.csv")

df_table

,Camden Kerbside Urban Traffic,London Marylebone Road Urban Traffic,Westminster - Oxford Street Urban Traffic,Wandsworth - Putney High Street Urban Traffic,Camden - Euston Road Urban Traffic
Distance between stations,,,,,
1st nearest background station,London N. Kensington Urban Background (3.69 km),London Bloomsbury Urban Background (1.99 km),London Bloomsbury Urban Background (2.08 km),Wandsworth - WA9 Urban Background (0.18 km),London Bloomsbury Urban Background (0.66 km)
2nd nearest background station,London Bloomsbury Urban Background (4.20 km),London Westminster Urban Background (3.47 km),London Westminster Urban Background (2.58 km),London N. Kensington Urban Background (6.41 km),London Westminster Urban Background (3.71 km)
1st nearest traffic station,Camden High Street Urban Traffic (2.23 km),Camden High Street Urban Traffic (2.10 km),RBKC Knightsbridge (Kensington and Chelsea) Ur...,Richmond Upon Thames - Castelnau Urban Traffic...,Camden High Street Urban Traffic (1.70 km)
2nd nearest traffic station,Brent - ARK Franklin Primary Academy Urban Tra...,RBKC Knightsbridge (Kensington and Chelsea) Ur...,Camden High Street Urban Traffic (2.98 km),Wandsworth - Lavender Hill (Clapham Jct) Urban...,RBKC Knightsbridge (Kensington and Chelsea) Ur...


# Table 3

In [ ]:
scenarios_map = {
    "traffic_tm":"scen1", 
    "baseline_tm":"scen1_w/o_traf",
    "nearest_bg_plus_traffic":"scen2", 
    "nearest_bg_no_traffic":"scen2_w/o_traf",
    "multi_station_plus_traffic":"scen3", 
    "multi_station_no_traffic":"scen3_w/o_traf"
}


base_dir = Path(r"Datasets_to_train\nested_cv_results_pollutants")
stations = ["Wandsworth-PutneyHighStreet", "CamdenKerbside", "LondonMaryleboneRoad"]
pollutants = ["NO2", "PM10", "PM25","O3"]
scenarios = [ "scen1", "scen1_w/o_traf", "scen2",  "scen2_w/o_traf", "scen3",  "scen3_w/o_traf"]

def safe_name(text):
    return str(text).replace(" ", "_").replace("/", "_").replace("-", "_")
    
df_best_scen = pd.DataFrame()
for station in stations:
    for pollutant in pollutants:
        if station == "CamdenKerbside" and pollutant == "O3":
            continue
        if station == "Wandsworth-PutneyHighStreet" and (pollutant in ["O3", "PM25"]):
            continue

        file_path = os.path.join(base_dir,f"{safe_name(station)}_{safe_name(pollutant)}")
        df_best = pd.read_csv(os.path.join(file_path, f"Final_test_metrics_{station}_{pollutant}.csv"))
        df_best = df_best.rename(columns={"Final_test_RMSE": "RMSE"})
        df_best["scenario"] = df_best["scenario"].apply(lambda x: scenarios_map.get(x, x))
        df_best_scen = pd.concat([df_best_scen, df_best], ignore_index=True)


df_cost_wide = df_best_scen.pivot_table(
    index=["station", "pollutant"],
    columns="scenario",
    values=["RMSE"],
    aggfunc="first"
)

df_cost_wide.columns = [
    f"{metric}_{scenario}" for metric, scenario in df_cost_wide.columns
]

df_cost_wide = df_cost_wide.reset_index()
df_cost_wide = df_cost_wide[
    [
        "pollutant",
        "station",
        "RMSE_scen1_w/o_traf",
        "RMSE_scen1",
        "RMSE_scen2_w/o_traf",
        "RMSE_scen2",
        "RMSE_scen3_w/o_traf",
        "RMSE_scen3",
    ]
]
# % improvement: positivo = multi_station_plus_traffic is better than traffic_tm
df_cost_wide["RMSE_improvement_scen1_vs_scen1_w/o_traf_%"] = (
    (
        df_cost_wide["RMSE_scen1_w/o_traf"]
        - df_cost_wide["RMSE_scen1"]
    )
    / df_cost_wide["RMSE_scen1_w/o_traf"]
) * 100

df_cost_wide["RMSE_improvement_scen2_vs_scen2_w/o_traf_%"] = (
    (
        df_cost_wide["RMSE_scen2_w/o_traf"]
        - df_cost_wide["RMSE_scen2"]
    )
    / df_cost_wide["RMSE_scen2_w/o_traf"]
) * 100

df_cost_wide["RMSE_improvement_scen3_vs_scen3_w/o_traf_%"] = (
    (
        df_cost_wide["RMSE_scen3_w/o_traf"]
        - df_cost_wide["RMSE_scen3"]
    )
    / df_cost_wide["RMSE_scen3_w/o_traf"]
) * 100

df_cost_wide["RMSE_improvement_scen2_vs_scen1_%"] = (
    (
        df_cost_wide["RMSE_scen1"]
        - df_cost_wide["RMSE_scen2"]
    )
    / df_cost_wide["RMSE_scen1"]
) * 100

df_cost_wide["RMSE_improvement_scen3_vs_scen1_%"] = (
    (
        df_cost_wide["RMSE_scen1"]
        - df_cost_wide["RMSE_scen3"]
    )
    / df_cost_wide["RMSE_scen1"]
) * 100

df_cost_wide["RMSE_improvement_scen3_vs_scen2_%"] = (
    (
        df_cost_wide["RMSE_scen2"]
        - df_cost_wide["RMSE_scen3"]
    )
    / df_cost_wide["RMSE_scen2"]
) * 100

df_cost_wide = df_cost_wide[["pollutant"] + [c for c in df_cost_wide.columns if c != "pollutant"]]
df_cost_wide= df_cost_wide.round(2)

df_cost_wide.sort_values(by=["pollutant", "station"], inplace=True )
df_cost_wide.to_csv("Table3.csv", index=False)
df_cost_wide.reset_index(drop=True, inplace=True)
df_cost_wide

,pollutant,station,RMSE_scen1_w/o_traf,RMSE_scen1,RMSE_scen2_w/o_traf,RMSE_scen2,RMSE_scen3_w/o_traf,RMSE_scen3,RMSE_improvement_scen1_vs_scen1_w/o_traf_%,RMSE_improvement_scen2_vs_scen2_w/o_traf_%,RMSE_improvement_scen3_vs_scen3_w/o_traf_%,RMSE_improvement_scen2_vs_scen1_%,RMSE_improvement_scen3_vs_scen1_%,RMSE_improvement_scen3_vs_scen2_%
0,NO2,CamdenKerbside,11.43,11.27,8.94,8.86,7.50,7.16,1.46,0.91,4.63,21.34,36.48,19.25
1,NO2,LondonMaryleboneRoad,9.73,8.72,8.77,7.96,6.80,6.43,10.38,9.28,5.44,8.77,26.24,19.15
2,NO2,Wandsworth-PutneyHighStreet,11.66,11.52,9.81,9.37,9.45,8.80,1.21,4.53,6.86,18.68,23.61,6.06
3,O3,LondonMaryleboneRoad,10.58,11.20,8.04,8.21,7.22,6.91,-5.79,-2.12,4.21,26.69,38.25,15.77
4,PM10,CamdenKerbside,11.34,10.53,9.72,9.70,9.31,9.23,7.14,0.14,0.88,7.88,12.38,4.88
5,PM10,LondonMaryleboneRoad,8.36,8.43,5.85,5.84,5.57,5.52,-0.90,0.26,0.91,30.82,34.53,5.37
6,PM10,Wandsworth-PutneyHighStreet,12.09,12.05,10.35,10.33,10.44,10.51,0.33,0.13,-0.68,14.24,12.77,-1.71
7,PM25,CamdenKerbside,5.49,4.93,2.43,2.41,2.42,2.54,10.19,0.63,-5.14,51.04,48.45,-5.28
8,PM25,LondonMaryleboneRoad,4.38,4.54,3.37,3.32,3.46,3.38,-3.64,1.32,2.30,26.76,25.45,-1.79


Mean RMSE Scen2 and Scen3

In [4]:
print(round(float(df_cost_wide["RMSE_improvement_scen2_vs_scen1_%"].mean()),2))
print(round(float(df_cost_wide["RMSE_improvement_scen3_vs_scen1_%"].mean()),2))

22.91
28.68


# Table SM 5.

In [101]:
base_dir = Path(r"Datasets_to_train\nested_cv_results_pollutants")

scenarios_map = {
    "traffic_tm":"S1", 
    "baseline_tm":"S1-noT",
    "nearest_bg_plus_traffic":"S2", 
    "nearest_bg_no_traffic":"S2-noT",
    "multi_station_plus_traffic":"S3", 
    "multi_station_no_traffic":"S3-noT"
}

folds_files = [
    p
    for p in Path(
        os.path.join(
            dir_files,
            "nested_cv_results_pollutants"
        )
    ).rglob("fold_metrics*.csv")
    if "partial_rerun_backups" not in p.parts
]

df_fold_metrics_all = pd.concat(
    [pd.read_csv(f) for f in folds_files],
    ignore_index=True
)
df_fold_metrics = df_fold_metrics_all[df_fold_metrics_all["model_name"] != "ridge"].copy()
df_perf = (
    df_fold_metrics
    .groupby(["station", "pollutant", "scenario", "model_name"])
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .reset_index()
)
df_perf.rename(columns={"NRMSE": "RMSE/σobs"}, inplace=True)
# Best model per station-pollutant-scenario
df_best_scen = df_perf.loc[
    df_perf.groupby(["pollutant", "station", "scenario"])["RMSE_mean"].idxmin()
].reset_index(drop=True)
df_best_scen= df_best_scen.round(2)

df_best_scen["scenario"] = df_best_scen["scenario"].apply(lambda x: scenarios_map.get(x, x))

station_order = ["LondonMaryleboneRoad", "CamdenKerbside", "Wandsworth-PutneyHighStreet", "Westminster-OxfordStreet", "Camden-EustonRoad"]
pollutant_order = ["NO2", "O3", "PM10", "PM25"]
scenario_order = ["S1", "S1-noT", "S2", "S2-noT", "S3", "S3-noT"]

# Sort by station -> pollutant -> scenario
df_best_scen["station"] = pd.Categorical(df_best_scen["station"], categories=station_order, ordered=True)
df_best_scen["pollutant"] = pd.Categorical(df_best_scen["pollutant"], categories=pollutant_order, ordered=True)
df_best_scen["scenario"] = pd.Categorical(df_best_scen["scenario"], categories=scenario_order, ordered=True)
df_best_scen = df_best_scen.sort_values(["station", "pollutant", "scenario"]).reset_index(drop=True)
df_best_scen = df_best_scen[["station", "pollutant", "scenario", "model_name", "RMSE_mean", "RMSE_std", "MAE_mean", "MAE_std", "R2_mean", "R2_std"]]


df_best_scen.to_csv("TableSM4_RMSE_MAE_R2_values.csv", index=False)


df_best_scen.head()


,station,pollutant,scenario,model_name,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,LondonMaryleboneRoad,NO2,S1,xgb,10.22,1.71,7.73,1.23,0.33,0.17
1,LondonMaryleboneRoad,NO2,S1-noT,lgbm,10.20,1.93,7.73,1.26,0.34,0.14
2,LondonMaryleboneRoad,NO2,S2,xgb,9.17,1.30,6.85,0.86,0.47,0.07
3,LondonMaryleboneRoad,NO2,S2-noT,etr,9.30,1.57,7.02,0.95,0.46,0.09
4,LondonMaryleboneRoad,NO2,S3,lgbm,8.01,1.43,5.96,1.06,0.59,0.09


Code to generate the table in latex

In [ ]:
csv_file = Path("TableSM4_RMSE_MAE_R2_values.csv")
tex_file = Path("TableSM4_RMSE_MAE_R2_values.tex")

df = pd.read_csv(csv_file)


station_labels = {"LondonMaryleboneRoad": "Marylebone Road", "CamdenKerbside": "Camden Kerbside", "Wandsworth-PutneyHighStreet": "Putney High Street", "Westminster-OxfordStreet": "Oxford Street", "Camden-EustonRoad": "Euston Road"}

pollutant_labels = {"NO2": r"NO$_2$", "O3": r"O$_3$", "PM10": r"PM$_{10}$", "PM25": r"PM$_{2.5}$"}

model_labels = {"rf": "RF", "etr": "ETR", "lgbm": "LGBM", "xgb": "XGB", "ridge": "Ridge"}

df["station"] = df["station"].map(station_labels).fillna(df["station"])
df["pollutant"] = df["pollutant"].map(pollutant_labels).fillna(df["pollutant"])
df["model_name"] = df["model_name"].astype(str).str.lower().map(model_labels).fillna(df["model_name"])


df["RMSE"] = df.apply(lambda row: f"{row['RMSE_mean']:.2f} $\\pm$ {row['RMSE_std']:.2f}", axis=1)
df["MAE"] = df.apply(lambda row: f"{row['MAE_mean']:.2f} $\\pm$ {row['MAE_std']:.2f}", axis=1)
df["R2"] = df.apply(lambda row: f"{row['R2_mean']:.2f} $\\pm$ {row['R2_std']:.2f}", axis=1)


latex_df = df[["pollutant", "station", "scenario", "model_name", "RMSE", "MAE", "R2"]].copy()

latex_df = latex_df.rename(columns={"pollutant": "Pollutant", "station": "Station", "scenario": "Scenario", "model_name": "Best model", "R2": r"$R^2$"})


pollutant_order = [r"NO$_2$", r"O$_3$", r"PM$_{10}$", r"PM$_{2.5}$"]
station_order = ["Marylebone Road", "Camden Kerbside", "Putney High Street", "Oxford Street", "Euston Road"]
scenario_order = ["S1", "S1-noT", "S2", "S2-noT", "S3", "S3-noT"]

latex_df["_pollutant_order"] = pd.Categorical(latex_df["Pollutant"], categories=pollutant_order, ordered=True)
latex_df["_station_order"] = pd.Categorical(latex_df["Station"], categories=station_order, ordered=True)
latex_df["_scenario_order"] = pd.Categorical(latex_df["Scenario"], categories=scenario_order, ordered=True)

latex_df = latex_df.sort_values(["_pollutant_order", "_station_order", "_scenario_order"]).drop(columns=["_pollutant_order", "_station_order", "_scenario_order"]).reset_index(drop=True)

tabular = latex_df.to_latex(index=False, escape=False, na_rep="--", column_format="llllccc")

latex_code = rf"""
\begin{{table}}[H]
\centering
\caption{{Performance of the best-performing ML model for each station--pollutant--scenario combination across the first four outer folds. Values are reported as mean $\pm$ standard deviation.}}
\label{{tab:SM4_model_performance}}
\resizebox{{\textwidth}}{{!}}{{%
{tabular}
}}
\end{{table}}
""".strip()

tex_file.write_text(latex_code, encoding="utf-8")

print(f"Saved: {tex_file.resolve()}")

# Table 8.

In [ ]:
scenarios_map = {
    "traffic_tm":"S1", 
    "baseline_tm":"S1-noT",
    "nearest_bg_plus_traffic":"S2", 
    "nearest_bg_no_traffic":"S2-noT",
    "multi_station_plus_traffic":"S3", 
    "multi_station_no_traffic":"S3-noT"
}
stations_map = {"Westminster-OxfordStreet":"Oxford St", "Camden-EustonRoad":"Euston",  }
dir_files = Path(r"Datasets_to_train")
metrics_files = list(Path("interpolation_results_chronological_selection").rglob("final_test_metrics*.csv"))
df_metrics_OK = pd.concat([pd.read_csv(f) for f in metrics_files], ignore_index=True)
df_metrics_OK = df_metrics_OK[(df_metrics_OK["station"] == "Camden-EustonRoad") | 
                                                 (df_metrics_OK["station"] == "Westminster-OxfordStreet")]
df_metrics_OK = df_metrics_OK[df_metrics_OK["pollutant"]=="NO2"]
df_metrics_OK = df_metrics_OK[["station","RMSE"]].rename(columns={"RMSE": "RMSE_OK"})


metrics_files = list(Path("interpolation_results_IDW").rglob("final_test_metrics*.csv"))
df_metrics_IDW = pd.concat( [pd.read_csv(f) for f in metrics_files], ignore_index=True)
df_metrics_IDW = df_metrics_IDW[(df_metrics_IDW["station"] == "Camden-EustonRoad") | 
                                                 (df_metrics_IDW["station"] == "Westminster-OxfordStreet")]
df_metrics_IDW = df_metrics_IDW[df_metrics_IDW["pollutant"]=="NO2"]
df_metrics_IDW = df_metrics_IDW[["station","RMSE"]].rename(columns={"RMSE": "RMSE_IDW"})

ML_dir = r"Datasets_to_train\nested_cv_results_pollutants"
metrics_files = list(Path(ML_dir).rglob("final_test_metrics*.csv"))
df_metrics_local = pd.concat([pd.read_csv(f) for f in metrics_files],ignore_index=True)
df_metrics_local = df_metrics_local[(df_metrics_local["station"] == "Camden-EustonRoad") | 
                                                 (df_metrics_local["station"] == "Westminster-OxfordStreet")]
df_metrics_local = df_metrics_local[df_metrics_local["pollutant"]=="NO2"]
df_metrics_local = df_metrics_local[["station","scenario","Final_test_RMSE"]].rename(columns={"Final_test_RMSE": "RMSE_local", "selected_model_name": "model_name"})


metrics_files = list(Path(ML_dir).rglob("ridge_final_test_metrics*.csv"))
df_metrics_ridge = pd.concat([pd.read_csv(f) for f in metrics_files], ignore_index=True)
df_metrics_ridge = df_metrics_ridge[(df_metrics_ridge["station"] == "Camden-EustonRoad") | 
                                                 (df_metrics_ridge["station"] == "Westminster-OxfordStreet")]
df_metrics_ridge = df_metrics_ridge[df_metrics_ridge["pollutant"]=="NO2"]
df_metrics_ridge = df_metrics_ridge[["station","scenario", "Final_test_RMSE"]].rename(columns={"Final_test_RMSE": "RMSE_ridge", "selected_model_name": "model_name"})


metrics_files = list(Path("validation_results").rglob("final_metrics_transfer*.csv"))
df_metrics_crossmodel = pd.concat( [pd.read_csv(f) for f in metrics_files],ignore_index=True)
df_metrics_crossmodel = df_metrics_crossmodel[(df_metrics_crossmodel["station"] == "Camden-EustonRoad") | 
                                                 (df_metrics_crossmodel["station"] == "Westminster-OxfordStreet")]
df_metrics_crossmodel = df_metrics_crossmodel[df_metrics_crossmodel["pollutant"]=="NO2"]
df_metrics_crossmodel = df_metrics_crossmodel[["station","scenario","RMSE"]].rename(columns={"RMSE": "RMSE_cross_site", "selected_model_name": "model_name"})


df_degradation = df_metrics_crossmodel.merge( df_metrics_local, on=["station", "scenario"], how="inner")
df_degradation = df_degradation.merge( df_metrics_ridge, on=["station", "scenario"],  how="left")
df_degradation = df_degradation.merge( df_metrics_IDW, on=["station"], how="left")
df_degradation = df_degradation.merge( df_metrics_OK, on=["station"], how="left")

df_degradation["RMSE_local"] = df_degradation["RMSE_local"].round(2)
df_degradation["RMSE_cross_site"] = df_degradation["RMSE_cross_site"].round(2)

df_degradation["degradation cross_site vs local"] = (
    (df_degradation["RMSE_cross_site"] - df_degradation["RMSE_local"])
    / df_degradation["RMSE_local"]
) * 100

df_degradation["RMSE_ridge"] = df_degradation["RMSE_ridge"].round(2)
df_degradation["degradation cross_site vs ridge"] = (
    (df_degradation["RMSE_cross_site"] - df_degradation["RMSE_ridge"])
    / df_degradation["RMSE_ridge"]
) * 100

df_degradation["degradation cross_site vs IDW"] = (
    (df_degradation["RMSE_cross_site"] - df_degradation["RMSE_IDW"])
    / df_degradation["RMSE_IDW"]
) * 100

df_degradation["degradation cross_site vs OK"] = (
    (df_degradation["RMSE_cross_site"] - df_degradation["RMSE_OK"])
    / df_degradation["RMSE_OK"]
) * 100

df_degradation = df_degradation.round(2)
df_degradation["scenario"] = df_degradation["scenario"].map(scenarios_map)
df_degradation["station"] = df_degradation["station"].map(stations_map)

df_degradation.to_csv("Table4.csv", index=False)

df_degradation

,station,scenario,RMSE_cross_site,RMSE_local,RMSE_ridge,RMSE_IDW,RMSE_OK,degradation cross_site vs local,degradation cross_site vs ridge,degradation cross_site vs IDW,degradation cross_site vs OK
0,Euston,S1,12.59,14.33,14.39,18.67,16.09,-12.14,-12.51,-32.57,-21.74
1,Euston,S1-noT,13.58,12.84,13.09,18.67,16.09,5.76,3.74,-27.26,-15.58
2,Euston,S2,12.24,13.20,13.85,18.67,16.09,-7.27,-11.62,-34.44,-23.91
3,Euston,S2-noT,12.67,13.11,12.10,18.67,16.09,-3.36,4.71,-32.14,-21.24
4,Euston,S3,10.60,15.28,12.62,18.67,16.09,-30.63,-16.01,-43.22,-34.11
5,Euston,S3-noT,11.03,12.98,10.03,18.67,16.09,-15.02,9.97,-40.92,-31.43
6,Oxford St,S1,15.19,10.74,12.48,13.01,13.12,41.43,21.71,16.76,15.76
7,Oxford St,S1-noT,15.62,11.42,13.38,13.01,13.12,36.78,16.74,20.06,19.04
8,Oxford St,S2,13.31,9.01,9.36,13.01,13.12,47.72,42.20,2.31,1.43
9,Oxford St,S2-noT,14.02,9.88,10.41,13.01,13.12,41.90,34.68,7.76,6.84


Code to generate the table in Latex

In [ ]:

csv_file = Path("Table4.csv")
tex_file = Path("Table4.tex")

df = pd.read_csv(csv_file)

latex_df = df[["station", "scenario",  "RMSE_cross_site","RMSE_local", "RMSE_ridge", "RMSE_IDW", "RMSE_OK", "degradation cross_site vs local", "degradation cross_site vs ridge", "degradation cross_site vs IDW", "degradation cross_site vs OK"]].copy()

latex_df = latex_df.rename(columns={"station": "Station", "scenario": "Scenario", "RMSE_local": "Local RMSE", "RMSE_cross_site": "Cross-site RMSE", "RMSE_ridge": "Ridge RMSE", "RMSE_IDW": "IDW RMSE", "RMSE_OK": "OK RMSE", 
                                    "degradation cross_site vs local": "Cross-site vs local", "degradation cross_site vs ridge": "Cross-site vs Ridge", "degradation cross_site vs IDW": "Cross-site vs IDW", "degradation cross_site vs OK": "Cross-site vs OK"})

station_order = ["Euston", "Oxford St"]
scenario_order = ["S1", "S1-noT", "S2", "S2-noT", "S3", "S3-noT"]

latex_df["_station_order"] = pd.Categorical(latex_df["Station"], categories=station_order, ordered=True)
latex_df["_scenario_order"] = pd.Categorical(latex_df["Scenario"], categories=scenario_order, ordered=True)

latex_df = latex_df.sort_values(["_station_order", "_scenario_order"]).drop(columns=["_station_order", "_scenario_order"]).reset_index(drop=True)

numeric_cols = ["Cross-site RMSE", "Local RMSE", "Ridge RMSE", "IDW RMSE", "OK RMSE", "Cross-site vs local", "Cross-site vs Ridge", "Cross-site vs IDW", "Cross-site vs OK"]
latex_df[numeric_cols] = latex_df[numeric_cols].round(2)

latex_df = latex_df.rename(columns={"Cross-site vs local": r"$\Delta$ Cross-site vs local (\%)", "Cross-site vs Ridge": r"$\Delta$ Cross-site vs Ridge (\%)", "Cross-site vs IDW": r"$\Delta$ Cross-site vs IDW (\%)", "Cross-site vs OK": r"$\Delta$ Cross-site vs OK (\%)"})

tabular = latex_df.to_latex(index=False, escape=False, na_rep="--", float_format="%.2f", column_format="llrrrrrrrrr")

latex_code = rf"""
\begin{{table}}[H]
\centering
\caption{{Comparison of the local and cross-site ML models with Ridge regression, inverse distance weighting (IDW), and Ordinary Kriging (OK) for NO$_2$ estimation.}}
\label{{tab:cross_site_comparison}}
\resizebox{{\textwidth}}{{!}}{{%
{tabular}
}}
\begin{{minipage}}{{\textwidth}}
\footnotesize
\textit{{Note:}} Positive $\Delta$ values indicate that the cross-site model had a higher RMSE than the corresponding comparison method, whereas negative values indicate a lower RMSE.
\end{{minipage}}
\end{{table}}
""".strip()

tex_file.write_text(latex_code, encoding="utf-8")

print(f"Saved: {tex_file.resolve()}")

# Table SM 6
This cell takes between 15 to 20 minutes to run and requires the trained models.

In [ ]:
base_dir = Path(r"Datasets_to_train\nested_cv_results_pollutants")

dir_files = r"Datasets_to_train"
pollutants = ["NO2","O3","PM10", "PM25"]
stations = ["LondonMaryleboneRoad", "CamdenKerbside", "Wandsworth-PutneyHighStreet", "Westminster-OxfordStreet", "Camden-EustonRoad"]

traffic_scenarios = [
    "traffic_tm",
    "nearest_bg_plus_traffic",
    "multi_station_plus_traffic"]

map_scenarios = {
    "traffic_tm":"Scen 1",
    "nearest_bg_plus_traffic":"Scen 2",
    "multi_station_plus_traffic":"Scen 3"
}

TRAFFIC_VARS = [ "traffic_level", "currenttraveltime"]

TEMPORAL_VARS = ["hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "month_sin", "month_cos"]

MET_VARS = [
    "temp_ow",
    "pressure_ow",
    "humidity_ow",
    "clouds_ow",
    "wind_speed_ow",
    "wdr_sin", "wdr_cos",
    "dew_point_ow",
]


def load_model(model_name, model_dir, station_results, pollutant, scenario):
    if model_name == "xgb":
        model_path = os.path.join(
            model_dir,
            f"{model_name}_{station_results}_{pollutant}_{scenario}_final.json"
        )
        model = XGBRegressor()
        model.load_model(model_path)

    elif model_name == "lgbm":
        model_path = os.path.join(
            model_dir,
            f"{model_name}_{station_results}_{pollutant}_{scenario}_final.txt"
        )
        model = lgb.Booster(model_file=model_path)

    elif model_name in ["etr", "rf"]:
        model_path = os.path.join(
            model_dir,
            f"{model_name}_{station_results}_{pollutant}_{scenario}_final.joblib"
        )
        model = joblib.load(model_path)

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    return model, model_path
all_group_summaries = []
for pollutant in pollutants:
    for j,station in enumerate(stations):

        if station == "Wandsworth-PutneyHighStreet":
            station_results = "Wandsworth_PutneyHighStreet"
        elif station == "Westminster-OxfordStreet":
            station_results = "Westminster_OxfordStreet"
        elif station == "Camden-EustonRoad":
            station_results = "Camden_EustonRoad"  
        else:
            station_results = station

        if station == "CamdenKerbside" and pollutant == "O3":
            continue
        elif station == "Wandsworth-PutneyHighStreet" and pollutant in ["PM25", "O3"]:
            continue
        elif station == "Westminster-OxfordStreet" and pollutant in ["PM10","PM25", "O3"]:
            continue
        elif station == "Camden-EustonRoad" and pollutant in ["PM10","PM25", "O3"]:
            continue
        
        if station == "Wandsworth-PutneyHighStreet":
            station_label = "PutneyHighStreet"
        elif station == "LondonMaryleboneRoad":
            station_label = "MaryleboneRoad"
        elif station == "CamdenKerbside":
            station_label = "CamdenKerbside"
        for i, scenario in enumerate(traffic_scenarios):

            print(pollutant, station, scenario)

            test_metrics = pd.read_csv(os.path.join(base_dir,f"{safe_name(station)}_{safe_name(pollutant)}",f"final_test_metrics_{station}_{pollutant}.csv"))
            model_name = test_metrics[(test_metrics["scenario"]==scenario)].iloc[0]["selected_model_name"]
            df_case = pd.read_csv(os.path.join(dir_files, f"{station}.csv"))
            case_dir = os.path.join(
                base_dir,
                f"{station_results}_{pollutant}"
            )
            feature_sets = build_scenario_feature_sets(df_case, target_col=pollutant)

            feature_cols = feature_sets[scenario]

            X_shap = df_case[feature_cols].copy()
            X_shap = X_shap.apply(pd.to_numeric, errors="coerce")

            model_dir = os.path.join(case_dir, "models_final")

            model, model_path = load_model(
                model_name=model_name,
                model_dir=model_dir,
                    station_results=station_results,
                    pollutant=pollutant,
                    scenario=scenario
                )
            if scenario == "traffic_tm":
                groups = {
                    "traffic": TRAFFIC_VARS,
                    "meteorology": MET_VARS,
                    "temporal": TEMPORAL_VARS
                }
            else:
                groups = {
                    "traffic": TRAFFIC_VARS,
                    "meteorology": MET_VARS,
                    "temporal": TEMPORAL_VARS,
                    "auxiliary_pollution": [
                        c for c in X_shap.columns
                        if "_bkg" in c or "_trf" in c
                    ]
                }

            
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_shap)
            shap_importance = pd.DataFrame({
                    "feature": X_shap.columns,
                    "mean_abs_shap": abs(shap_values).mean(axis=0)
                }).sort_values("mean_abs_shap", ascending=False)

            group_summary = []

            total_importance = shap_importance["mean_abs_shap"].sum()

            for group_name, vars_list in groups.items():

                group_value = (
                    shap_importance[
                        shap_importance["feature"].isin(vars_list)
                    ]["mean_abs_shap"]
                    .sum()
                )

                group_summary.append({
                    "group": group_name,
                    "importance": round(group_value,2),
                    "percentage": round((group_value / total_importance) * 100,2)
                })

            df_group_summary = pd.DataFrame(group_summary)

            df_group_summary["pollutant"] = pollutant
            df_group_summary["station"] = station
            df_group_summary["scenario"] = scenario
            df_group_summary["model_name"] = model_name

            all_group_summaries.append(df_group_summary)
            if not os.path.exists(model_path):
                print("File does not exist:", model_path)
                continue
df_shap_group_summary_all = pd.concat(
    all_group_summaries,
    ignore_index=True
)
df_shap_group_summary_all["scenario"] = (df_shap_group_summary_all["scenario"].map(map_scenarios))

df_shap_group_summary_all[
    ["importance", "percentage"]
] = df_shap_group_summary_all[
    ["importance", "percentage"]
].round(2)
df_shap_group_summary_all = df_shap_group_summary_all[
    [
        "pollutant",
        "station",
        "scenario",
        "model_name",
        "group",
        "importance",
        "percentage",
    ]
]

df_shap_percentages = (
    df_shap_group_summary_all
    .pivot_table(
        index=["pollutant", "station", "scenario", "model_name"],
        columns="group",
        values="percentage",
        aggfunc="mean"
    )
    .reset_index()
)

df_shap_percentages = df_shap_percentages.rename(columns={
    "temporal": "Percentage of temporal variables",
    "traffic": "Percentage of traffic variables",
    "meteorology": "Percentage of meteorological variables",
    "auxiliary_pollution": "Percentage of auxiliary pollution variables",
})

cols_to_round = [
    "Percentage of temporal variables",
    "Percentage of traffic variables",
    "Percentage of meteorological variables",
    "Percentage of auxiliary pollution variables"
]

df_shap_percentages[cols_to_round] = df_shap_percentages[cols_to_round].round(2)
scenario_order = [
    "Scen 1",
    "Scen 2",
    "Scen 3"
]

invalid_mask = (
    ((df_shap_percentages["station"] == "CamdenKerbside") &
     (df_shap_percentages["pollutant"] == "O3")) |
    ((df_shap_percentages["station"] == "Wandsworth-PutneyHighStreet") &
     (df_shap_percentages["pollutant"].isin(["PM25", "O3"]))) |
    ((df_shap_percentages["station"] == "Westminster-OxfordStreet") &
     (df_shap_percentages["pollutant"].isin(["PM10", "PM25", "O3"]))) |
    ((df_shap_percentages["station"] == "Camden-EustonRoad") &
     (df_shap_percentages["pollutant"].isin(["PM10", "PM25", "O3"])))
)

df_shap_percentages = df_shap_percentages[~invalid_mask].reset_index(drop=True)
df_shap_percentages = df_shap_percentages[
    [
        "pollutant",
        "station",
        "scenario",
        "model_name",
        "Percentage of temporal variables",
        "Percentage of meteorological variables",
        "Percentage of traffic variables",
        "Percentage of auxiliary pollution variables"
    ]
]

df_shap_percentages["scenario"] = pd.Categorical(
    df_shap_percentages["scenario"],
    categories=scenario_order,
    ordered=True
)
df_shap_percentages = df_shap_percentages.sort_values(
    ["pollutant", "station", "scenario"]
).reset_index(drop=True)
df_shap_percentages.to_csv("Table SM5 df_shap_percentages.csv", index=False)

df_shap_latex = df_shap_percentages.copy()

station_labels = {
    "LondonMaryleboneRoad": "Marylebone Road",
    "CamdenKerbside": "Camden Kerbside",
    "Wandsworth-PutneyHighStreet": "Putney High Street",
    "Westminster-OxfordStreet": "Oxford Street",
    "Camden-EustonRoad": "Euston Road",
}

model_labels = {
    "rf": "RF",
    "etr": "ETR",
    "lgbm": "LGBM",
    "xgb": "XGB",
}

pollutant_labels = {
    "NO2": r"NO$_2$",
    "O3": r"O$_3$",
    "PM10": r"PM$_{10}$",
    "PM25": r"PM$_{2.5}$",
}

df_shap_latex["station"] = (
    df_shap_latex["station"]
    .map(station_labels)
    .fillna(df_shap_latex["station"])
)

df_shap_latex["model_name"] = (
    df_shap_latex["model_name"]
    .astype(str)
    .str.lower()
    .map(model_labels)
    .fillna(df_shap_latex["model_name"])
)

df_shap_latex["pollutant"] = (
    df_shap_latex["pollutant"]
    .map(pollutant_labels)
    .fillna(df_shap_latex["pollutant"])
)

df_shap_latex = df_shap_latex.rename(
    columns={
        "pollutant": "Pollutant",
        "station": "Station",
        "scenario": "Scenario",
        "model_name": "Model",
        "Percentage of temporal variables": "Temporal (%)",
        "Percentage of meteorological variables": "Meteorological (%)",
        "Percentage of traffic variables": "Traffic (%)",
        "Percentage of auxiliary pollution variables": "N-S pollutants (%)",
    }
)

percentage_columns = [
    "Temporal (%)",
    "Meteorological (%)",
    "Traffic (%)",
    "N-S pollutants (%)",
]

for column in percentage_columns:
    df_shap_latex[column] = pd.to_numeric(
        df_shap_latex[column],
        errors="coerce",
    )

latex_code = df_shap_latex.to_latex(
    index=False,
    escape=False,
    na_rep="--",
    float_format="%.2f",
    column_format="llllrrrr",
    caption=(
        "Percentage contribution of feature groups to the model predictions, grouped by pollutant, "
        "station, and scenario. N-S pollutants refer to the pollutants "
        "measured at the neighbouring stations from the corresponding scenario and the site to be estimated."
    ),
    label="tab:Percentage_contribution",
    position="htbp",
)

latex_path = Path("Table_SM5_df_shap_percentages.tex")

latex_path.write_text(
    latex_code,
    encoding="utf-8",
)

print(f"CSV table saved to: Table SM5 df_shap_percentages.csv")
print(f"LaTeX table saved to: {latex_path}")

df_shap_percentages

NO2 LondonMaryleboneRoad traffic_tm
NO2 LondonMaryleboneRoad nearest_bg_plus_traffic
NO2 LondonMaryleboneRoad multi_station_plus_traffic
NO2 CamdenKerbside traffic_tm
NO2 CamdenKerbside nearest_bg_plus_traffic
NO2 CamdenKerbside multi_station_plus_traffic
NO2 Wandsworth-PutneyHighStreet traffic_tm
NO2 Wandsworth-PutneyHighStreet nearest_bg_plus_traffic
NO2 Wandsworth-PutneyHighStreet multi_station_plus_traffic
NO2 Westminster-OxfordStreet traffic_tm
NO2 Westminster-OxfordStreet nearest_bg_plus_traffic
NO2 Westminster-OxfordStreet multi_station_plus_traffic
NO2 Camden-EustonRoad traffic_tm
NO2 Camden-EustonRoad nearest_bg_plus_traffic
NO2 Camden-EustonRoad multi_station_plus_traffic
O3 LondonMaryleboneRoad traffic_tm
O3 LondonMaryleboneRoad nearest_bg_plus_traffic
O3 LondonMaryleboneRoad multi_station_plus_traffic
PM10 LondonMaryleboneRoad traffic_tm
PM10 LondonMaryleboneRoad nearest_bg_plus_traffic
PM10 LondonMaryleboneRoad multi_station_plus_traffic
PM10 CamdenKerbside traffic_tm
PM10

group,pollutant,station,scenario,model_name,Percentage of temporal variables,Percentage of meteorological variables,Percentage of traffic variables,Percentage of auxiliary pollution variables
0,NO2,Camden-EustonRoad,Scen 1,etr,36.58,37.14,26.28,NaN
1,NO2,Camden-EustonRoad,Scen 2,xgb,18.62,44.95,25.57,10.85
2,NO2,Camden-EustonRoad,Scen 3,lgbm,9.08,22.54,13.97,54.42
3,NO2,CamdenKerbside,Scen 1,xgb,19.64,55.78,24.58,NaN
4,NO2,CamdenKerbside,Scen 2,xgb,18.70,32.64,19.11,29.55
5,NO2,CamdenKerbside,Scen 3,lgbm,11.02,17.16,13.05,58.77
6,NO2,LondonMaryleboneRoad,Scen 1,xgb,30.03,47.78,22.19,NaN
7,NO2,LondonMaryleboneRoad,Scen 2,xgb,24.00,43.09,17.04,15.87
8,NO2,LondonMaryleboneRoad,Scen 3,lgbm,16.04,23.66,11.88,48.41
9,NO2,Wandsworth-PutneyHighStreet,Scen 1,xgb,18.68,55.07,26.26,NaN


# Tables 9-11

In [ ]:
OUTPUT_DIR = Path("block_bootstrap_results")
ARTICLE_TABLES_DIR = OUTPUT_DIR / "article_tables"
ARTICLE_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTICLE_COMPARISONS = [
    "scen1 vs scen1-noT",
    "scen2 vs scen2-noT",
    "scen3 vs scen3-noT",
]

COMPARISON_FILE_LABELS = {
    "scen1 vs scen1-noT": "scen1_vs_scen1_noT",
    "scen2 vs scen2-noT": "scen2_vs_scen2_noT",
    "scen3 vs scen3-noT": "scen3_vs_scen3_noT",
}

COMPARISON_CAPTIONS = {
    "scen1 vs scen1-noT": (
        "Paired 24-hour block-bootstrap comparison between "
        "Scen~1 and Scen~1-noT."
    ),
    "scen2 vs scen2-noT": (
        "Paired 24-hour block-bootstrap comparison between "
        "Scen~2 and Scen~2-noT."
    ),
    "scen3 vs scen3-noT": (
        "Paired 24-hour block-bootstrap comparison between "
        "Scen~3 and Scen~3-noT."
    ),
}

COMPARISON_LATEX_LABELS = {
    "scen1 vs scen1-noT": "tab:bootstrap_scen1",
    "scen2 vs scen2-noT": "tab:bootstrap_scen2",
    "scen3 vs scen3-noT": "tab:bootstrap_scen3",
}

STATION_LABELS = {
    "LondonMaryleboneRoad": "Marylebone Road",
    "CamdenKerbside": "Camden Kerbside",
    "Wandsworth-PutneyHighStreet": "Putney High Street",
    "Westminster-OxfordStreet": "Oxford Street",
    "Camden-EustonRoad": "Euston Road",
}

POLLUTANT_LABELS_CSV = {
    "NO2": "NO2",
    "O3": "O3",
    "PM10": "PM10",
    "PM25": "PM2.5",
}

POLLUTANT_LABELS_LATEX = {
    "NO2": r"NO$_2$",
    "O3": r"O$_3$",
    "PM10": r"PM$_{10}$",
    "PM25": r"PM$_{2.5}$",
}

MODEL_LABELS = {
    "rf": "RF",
    "etr": "ETR",
    "lgbm": "LGBM",
    "xgb": "XGB",
}

STATION_ORDER = [
    "LondonMaryleboneRoad",
    "CamdenKerbside",
    "Wandsworth-PutneyHighStreet",
    "Westminster-OxfordStreet",
    "Camden-EustonRoad",
]

POLLUTANT_ORDER = [
    "NO2",
    "O3",
    "PM10",
    "PM25",
]


def format_pvalue(value: float) -> str:
    """
    Formats p-values for the article table.
    """

    if pd.isna(value):
        return "--"

    if value < 0.001:
        return "<0.001"

    return f"{value:.3f}"


def format_ci(
    ci_low: float,
    ci_high: float,
) -> str:
    """
    Formats the relative RMSE confidence interval.
    """

    if pd.isna(ci_low) or pd.isna(ci_high):
        return "--"

    return f"[{ci_low:.2f}, {ci_high:.2f}]"


def yes_no(value) -> str:
    """
    Converts Boolean significance values to Yes/No.
    """

    if pd.isna(value):
        return "--"

    return "Yes" if bool(value) else "No"


def escape_latex_text(text: str) -> str:
    """
    Escapes selected characters used in ordinary LaTeX text.
    """

    return (
        str(text)
        .replace("&", r"\&")
        .replace("%", r"\%")
        .replace("_", r"\_")
        .replace("#", r"\#")
    )


def create_article_table(
    results_df: pd.DataFrame,
    comparison_name: str,
) -> pd.DataFrame:
    """
    Creates one article table for one scenario comparison.

    Only RMSE results are retained.

    The reported difference is:

        RMSE_with_traffic - RMSE_without_traffic

    Therefore, negative values indicate a reduction in RMSE
    after including traffic-related variables.
    """

    table = results_df.loc[
        (results_df["metric"] == "RMSE")
        & (results_df["comparison"] == comparison_name)
    ].copy()

    if table.empty:
        raise ValueError(
            f"No RMSE results found for comparison: "
            f"{comparison_name}"
        )

    duplicated = table.duplicated(
        subset=[
            "station",
            "pollutant",
            "comparison",
            "metric",
        ],
        keep=False,
    )

    if duplicated.any():
        duplicate_rows = table.loc[
            duplicated,
            [
                "station",
                "pollutant",
                "comparison",
                "metric",
            ],
        ]

        raise ValueError(
            "Duplicated rows found in article table:\n"
            f"{duplicate_rows}"
        )

    table["Best model"] = (
        table["selected_model_b"]
        .astype(str)
        .str.lower()
        .map(MODEL_LABELS)
        .fillna(table["selected_model_b"])
    )

    table["Station"] = (
        table["station"]
        .map(STATION_LABELS)
        .fillna(table["station"])
    )

    table["Pollutant"] = (
        table["pollutant"]
        .map(POLLUTANT_LABELS_CSV)
        .fillna(table["pollutant"])
    )

    table["Delta RMSE (%)"] = (
        table["relative_difference_percent"]
        .round(2)
    )

    table["Delta RMSE (%)"] = (
        table["relative_increment_percent"]
        .round(2)
    )
    table["95% CI"] = [
        format_ci(low, high)
        for low, high in zip(
            table["relative_difference_ci_low"],
            table["relative_difference_ci_high"],
        )
    ]

    table["Holm-adjusted p-value"] = (
        table["pvalue_holm_within_case"]
        .apply(format_pvalue)
    )

    table["Significant (CI)"] = (
        table["significant_ci"]
        .apply(yes_no)
    )

    table["Significant (Holm)"] = (
        table["significant_holm_within_case"]
        .apply(yes_no)
    )
    table["Significant improvement"] = np.where(
        (table["relative_improvement_percent"] > 0)
        & (
            table["significant_ci"]
            | table["significant_holm_within_case"]
        ),
        "Yes",
        "No",
    )
    table["_pollutant_order"] = pd.Categorical(
    table["pollutant"],
    categories=POLLUTANT_ORDER,
    ordered=True,
    )

    table["_station_order"] = pd.Categorical(
        table["station"],
        categories=STATION_ORDER,
        ordered=True,
    )

    table = (
        table
        .sort_values(
            [
                "_pollutant_order",
                "_station_order",
            ]
        )
        .reset_index(drop=True)
    )
    final_columns = [
        "Pollutant",
        "Station",
        "Best model",
        "Delta RMSE (%)",
        "95% CI",
        "Holm-adjusted p-value",
        #"Significant (CI)",
        #"Significant (Holm)",
        "Significant improvement",
    ]

    return table[final_columns]


def table_to_latex(
    article_table: pd.DataFrame,
    comparison_name: str,
) -> str:
    """
    Creates a complete LaTeX table ready to include in Overleaf.
    """

    latex_table = article_table.copy()

    csv_to_latex_pollutant = {
        "NO2": r"NO$_2$",
        "O3": r"O$_3$",
        "PM10": r"PM$_{10}$",
        "PM2.5": r"PM$_{2.5}$",
    }

    latex_table["Pollutant"] = (
    latex_table["Pollutant"]
    .map(csv_to_latex_pollutant)
    .fillna(latex_table["Pollutant"])
    )

    latex_table["Delta RMSE (%)"] = (
        latex_table["Delta RMSE (%)"]
        .apply(
            lambda value: (
                "--"
                if pd.isna(value)
                else f"{value:.2f}"
            )
        )
    )

    tabular_code = latex_table.to_latex(
        index=False,
        escape=False,
        na_rep="--",
        column_format="lllrrrrr",
        header=[
            "Station",
            "Pollutant",
            "Best model",
            r"$\Delta$RMSE (\%)",
            "95\\% CI",
            "Holm-adjusted $p$-value",
            "Significant improvement"
        ],
    )

    caption = COMPARISON_CAPTIONS[comparison_name]
    label = COMPARISON_LATEX_LABELS[comparison_name]

    notes = (
        r"\textit{Note:} $\Delta$RMSE is calculated as "
        r"RMSE$_{\mathrm{with\ traffic}}-$"
        r"RMSE$_{\mathrm{without\ traffic}}$. "
        r"Therefore, negative values indicate that the inclusion of "
        r"traffic-related variables reduced the RMSE. "
        r"Confidence intervals and $p$-values were obtained using a "
        r"paired non-overlapping 24-hour block bootstrap with 10,000 "
        r"replicates. Holm-adjusted $p$-values correspond to the "
        r"within-station--pollutant correction."
    )

    complete_latex = f"""
\\begin{{table}}[htbp]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\resizebox{{\\textwidth}}{{!}}{{%
{tabular_code}
}}
\\begin{{minipage}}{{\\textwidth}}
\\footnotesize
{notes}
\\end{{minipage}}
\\end{{table}}
"""

    return complete_latex.strip() + "\n"


def save_article_tables(
    results_df: pd.DataFrame,
) -> None:
    """
    Saves one CSV and one LaTeX file per scenario comparison.
    """

    for comparison_name in ARTICLE_COMPARISONS:

        article_table = create_article_table(
            results_df=results_df,
            comparison_name=comparison_name,
        )

        file_label = COMPARISON_FILE_LABELS[
            comparison_name
        ]

        csv_path = (
            ARTICLE_TABLES_DIR
            / f"bootstrap_{file_label}.csv"
        )

        article_table.to_csv(
            csv_path,
            index=False,
        )

        latex_code = table_to_latex(
            article_table=article_table,
            comparison_name=comparison_name,
        )

        latex_path = (
            ARTICLE_TABLES_DIR
            / f"bootstrap_{file_label}.tex"
        )

        latex_path.write_text(
            latex_code,
            encoding="utf-8",
        )

        print(
            f"\nSaved article table for "
            f"{comparison_name}:",
            flush=True,
        )
        print(f"  CSV:   {csv_path}", flush=True)
        print(f"  LaTeX: {latex_path}", flush=True)

        print("\nPreview:")
        print(article_table.to_string(index=False))

results = pd.read_csv(r"block_bootstrap_results\block_bootstrap_summary.csv")

scenarios_map = {
"multi_station_plus_traffic vs multi_station_no_traffic": "scen3 vs scen3-noT",
"nearest_bg_plus_traffic vs nearest_bg_no_traffic": "scen2 vs scen2-noT",
"traffic_tm vs baseline_tm": "scen1 vs scen1-noT",
"nearest_bg_plus_traffic vs traffic_tm": "scen2 vs scen1",
"multi_station_plus_traffic vs nearest_bg_plus_traffic": "scen3 vs scen2",
"multi_station_plus_traffic vs traffic_tm": "scen3 vs scen1"
}

results["comparison"] = results["comparison"].apply(
    lambda x: scenarios_map.get(x, x)
)

results["difference_a_minus_b"] = -results["difference_b_minus_a"]
results["relative_improvement_percent"] = (
    -results["relative_difference_percent"]
)

results["improvement_ci_low"] = (
    -results["relative_difference_ci_high"]
)

results["improvement_ci_high"] = (
    -results["relative_difference_ci_low"]
)
results["significant_improvement_holm"] = np.where(
    (results["relative_improvement_percent"] > 0)
    & results["significant_holm_within_case"],
    "Yes",
    "No",
)
save_article_tables(results)


compact_columns = [
    "station", "pollutant", "comparison", "scenario_a", "scenario_b",
    "selected_model_a", "selected_model_b", "metric", "value_a", "value_b",
    "difference_b_minus_a", "difference_ci_low", "difference_ci_high",
    "relative_difference_percent", "relative_difference_ci_low",
    "relative_difference_ci_high", "probability_b_better", "pvalue_raw",
    "pvalue_holm_within_case", "pvalue_holm_global", "significant_ci",
    "significant_holm_within_case", "significant_holm_global",
    "n_observations", "n_blocks", "block_hours", "n_bootstrap",
]


results[compact_columns].to_csv(
    OUTPUT_DIR / "block_bootstrap_summary_compact.csv",
    index=False,
)

save_article_tables(results)



Saved article table for scen1 vs scen1-noT:
  CSV:   block_bootstrap_results\article_tables\bootstrap_scen1_vs_scen1_noT.csv
  LaTeX: block_bootstrap_results\article_tables\bootstrap_scen1_vs_scen1_noT.tex

Preview:
Pollutant            Station Best model  Delta RMSE (%)          95% CI Holm-adjusted p-value Significant improvement
      NO2    Marylebone Road        XGB          -10.38 [-16.70, -0.56]                 0.058                     Yes
      NO2    Camden Kerbside        XGB           -1.46   [-3.08, 0.37]                 0.214                      No
      NO2 Putney High Street        XGB           -1.21   [-4.57, 2.07]                 0.471                      No
      NO2      Oxford Street        XGB           -5.95 [-10.18, -0.89]                 0.030                     Yes
      NO2        Euston Road        ETR           11.61   [1.96, 21.48]                 0.072                      No
       O3    Marylebone Road       LGBM            5.79   [0.85, 10.71]    

In [ ]:

INPUT_FILE = Path(
    r"block_bootstrap_results\block_bootstrap_summary.csv"
)

OUTPUT_DIR = Path(
    "block_bootstrap_results"
)

ARTICLE_TABLES_DIR = OUTPUT_DIR / "article_tables"
ARTICLE_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


ARTICLE_COMPARISONS = [
    "scen1 vs scen1-noT",
    "scen2 vs scen2-noT",
    "scen3 vs scen3-noT",
]


SCENARIOS_MAP = {
    (
        "multi_station_plus_traffic "
        "vs multi_station_no_traffic"
    ): "scen3 vs scen3-noT",

    (
        "nearest_bg_plus_traffic "
        "vs nearest_bg_no_traffic"
    ): "scen2 vs scen2-noT",

    "traffic_tm vs baseline_tm":
        "scen1 vs scen1-noT",

    (
        "nearest_bg_plus_traffic "
        "vs traffic_tm"
    ): "scen2 vs scen1",

    (
        "multi_station_plus_traffic "
        "vs nearest_bg_plus_traffic"
    ): "scen3 vs scen2",

    (
        "multi_station_plus_traffic "
        "vs traffic_tm"
    ): "scen3 vs scen1",
}


COMPARISON_FILE_LABELS = {
    "scen1 vs scen1-noT":
        "scen1_vs_scen1_noT",

    "scen2 vs scen2-noT":
        "scen2_vs_scen2_noT",

    "scen3 vs scen3-noT":
        "scen3_vs_scen3_noT",
}


COMPARISON_CAPTIONS = {
    "scen1 vs scen1-noT": (
        "Paired 24-hour block-bootstrap comparison between "
        "Scen~1 and Scen~1-noT."
    ),

    "scen2 vs scen2-noT": (
        "Paired 24-hour block-bootstrap comparison between "
        "Scen~2 and Scen~2-noT."
    ),

    "scen3 vs scen3-noT": (
        "Paired 24-hour block-bootstrap comparison between "
        "Scen~3 and Scen~3-noT."
    ),
}


COMPARISON_LATEX_LABELS = {
    "scen1 vs scen1-noT":
        "tab:bootstrap_scen1",

    "scen2 vs scen2-noT":
        "tab:bootstrap_scen2",

    "scen3 vs scen3-noT":
        "tab:bootstrap_scen3",
}


POLLUTANT_ORDER = [
    "NO2",
    "O3",
    "PM10",
    "PM25",
]


STATION_ORDER = [
    "LondonMaryleboneRoad",
    "CamdenKerbside",
    "Wandsworth-PutneyHighStreet",
    "Westminster-OxfordStreet",
    "Camden-EustonRoad",
]


POLLUTANT_LABELS_CSV = {
    "NO2": "NO2",
    "O3": "O3",
    "PM10": "PM10",
    "PM25": "PM2.5",
}


POLLUTANT_LABELS_LATEX = {
    "NO2": r"NO$_2$",
    "O3": r"O$_3$",
    "PM10": r"PM$_{10}$",
    "PM2.5": r"PM$_{2.5}$",
}


STATION_LABELS = {
    "LondonMaryleboneRoad":
        "Marylebone Road",

    "CamdenKerbside":
        "Camden Kerbside",

    "Wandsworth-PutneyHighStreet":
        "Putney High Street",

    "Westminster-OxfordStreet":
        "Oxford Street",

    "Camden-EustonRoad":
        "Euston Road",
}


MODEL_LABELS = {
    "rf": "RF",
    "etr": "ETR",
    "lgbm": "LGBM",
    "xgb": "XGB",
}


def format_pvalue(value: float) -> str:
    """Format Holm-adjusted p-values."""

    if pd.isna(value):
        return "--"

    if value < 0.001:
        return "<0.001"

    return f"{value:.3f}"


def format_ci(
    lower: float,
    upper: float,
) -> str:
    """Format a 95% confidence interval."""

    if pd.isna(lower) or pd.isna(upper):
        return "--"

    return f"[{lower:.2f}, {upper:.2f}]"


def prepare_bootstrap_results(
    results: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert the bootstrap results from the original convention:

        B - A

    to the manuscript convention:

        A - B

    where:
        A = scenario without traffic;
        B = corresponding scenario with traffic.

    Thus, positive Delta RMSE values indicate that adding
    traffic variables reduced the RMSE.
    """

    required_columns = {
        "station",
        "pollutant",
        "comparison",
        "metric",
        "selected_model_b",
        "relative_difference_percent",
        "relative_difference_ci_low",
        "relative_difference_ci_high",
        "pvalue_holm_within_case",
        "significant_holm_within_case",
    }

    missing_columns = required_columns.difference(
        results.columns
    )

    if missing_columns:
        raise ValueError(
            "The bootstrap file is missing columns: "
            f"{sorted(missing_columns)}"
        )

    result = results.copy()

    result["comparison"] = (
        result["comparison"]
        .map(SCENARIOS_MAP)
        .fillna(result["comparison"])
    )

    result["relative_improvement_percent"] = (
        -result["relative_difference_percent"]
    )

    result["improvement_ci_low"] = (
        -result["relative_difference_ci_high"]
    )

    result["improvement_ci_high"] = (
        -result["relative_difference_ci_low"]
    )

    result["significant_improvement_holm"] = (
        (result["relative_improvement_percent"] > 0)
        & result["significant_holm_within_case"].astype(bool)
    )

    return result



def create_article_table(
    results: pd.DataFrame,
    comparison_name: str,
) -> pd.DataFrame:
    """
    Create one RMSE table for a direct comparison between
    a scenario with traffic and its corresponding scenario
    without traffic.
    """

    table = results.loc[
        (results["metric"] == "RMSE")
        & (results["comparison"] == comparison_name)
    ].copy()

    if table.empty:
        raise ValueError(
            f"No RMSE results found for {comparison_name}."
        )

    duplicated = table.duplicated(
        subset=[
            "station",
            "pollutant",
            "comparison",
            "metric",
        ],
        keep=False,
    )

    if duplicated.any():
        duplicate_rows = table.loc[
            duplicated,
            [
                "station",
                "pollutant",
                "comparison",
                "metric",
            ],
        ]

        raise ValueError(
            "Duplicated rows found:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )

    table["Pollutant"] = (
        table["pollutant"]
        .map(POLLUTANT_LABELS_CSV)
        .fillna(table["pollutant"])
    )

    table["Station"] = (
        table["station"]
        .map(STATION_LABELS)
        .fillna(table["station"])
    )

    table["Best model"] = (
        table["selected_model_b"]
        .astype(str)
        .str.lower()
        .map(MODEL_LABELS)
        .fillna(table["selected_model_b"])
    )

    table["Delta RMSE (%)"] = (
        table["relative_improvement_percent"]
        .round(2)
    )

    table["95% CI"] = [
        format_ci(lower, upper)
        for lower, upper in zip(
            table["improvement_ci_low"],
            table["improvement_ci_high"],
        )
    ]

    table["Holm-adjusted p-value"] = (
        table["pvalue_holm_within_case"]
        .apply(format_pvalue)
    )

    table["Significant improvement"] = np.where(
        table["significant_improvement_holm"],
        "Yes",
        "No",
    )

    table["_pollutant_order"] = pd.Categorical(
        table["pollutant"],
        categories=POLLUTANT_ORDER,
        ordered=True,
    )

    table["_station_order"] = pd.Categorical(
        table["station"],
        categories=STATION_ORDER,
        ordered=True,
    )

    table = (
        table
        .sort_values(
            [
                "_pollutant_order",
                "_station_order",
            ]
        )
        .reset_index(drop=True)
    )

    final_columns = [
        "Pollutant",
        "Station",
        "Best model",
        "Delta RMSE (%)",
        "95% CI",
        "Holm-adjusted p-value",
        "Significant improvement",
    ]

    return table[final_columns]



def create_latex_table(
    article_table: pd.DataFrame,
    comparison_name: str,
) -> str:
    """
    Create a complete LaTeX table ready to include
    in Overleaf.
    """

    latex_table = article_table.copy()

    latex_table["Pollutant"] = (
        latex_table["Pollutant"]
        .map(POLLUTANT_LABELS_LATEX)
        .fillna(latex_table["Pollutant"])
    )

    latex_table["Delta RMSE (%)"] = (
        latex_table["Delta RMSE (%)"]
        .apply(
            lambda value: (
                "--"
                if pd.isna(value)
                else f"{value:.2f}"
            )
        )
    )

    latex_table = latex_table.rename(
        columns={
            "Best model":
                "Best model",

            "Delta RMSE (%)":
                r"$\Delta_{\mathrm{RMSE}}$ (\%)",

            "95% CI":
                r"95\% CI",

            "Holm-adjusted p-value":
                r"Holm-adjusted $p$-value",

            "Significant improvement":
                "Significant improvement",
        }
    )

    tabular_code = latex_table.to_latex(
        index=False,
        escape=False,
        na_rep="--",
        column_format="lllclcc",
    )

    caption = COMPARISON_CAPTIONS[
        comparison_name
    ]

    label = COMPARISON_LATEX_LABELS[
        comparison_name
    ]

    note = (
        r"\textit{Note:} "
        r"$\Delta_{\mathrm{RMSE}}(\%) = "
        r"100(\mathrm{RMSE}_{A}-\mathrm{RMSE}_{B})/"
        r"\mathrm{RMSE}_{A}$, where A represents the "
        r"scenario without traffic-related variables and B "
        r"represents the corresponding scenario including "
        r"traffic-related variables. Positive values indicate "
        r"that including traffic variables reduced the RMSE. "
        r"A statistically significant improvement is reported "
        r"as Yes only when $\Delta_{\mathrm{RMSE}}>0$ and the "
        r"Holm-adjusted $p$-value is below 0.05."
    )

    return (
        f"\\begin{{table}}[htbp]\n"
        f"\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        f"\\resizebox{{\\textwidth}}{{!}}{{%\n"
        f"{tabular_code}"
        f"}}\n"
        f"\\begin{{minipage}}{{\\textwidth}}\n"
        f"\\footnotesize\n"
        f"{note}\n"
        f"\\end{{minipage}}\n"
        f"\\end{{table}}\n"
    )


def save_article_tables(
    results: pd.DataFrame,
) -> None:
    """Save one CSV and one LaTeX file per comparison."""

    for comparison_name in ARTICLE_COMPARISONS:

        table = create_article_table(
            results=results,
            comparison_name=comparison_name,
        )

        file_label = COMPARISON_FILE_LABELS[
            comparison_name
        ]

        csv_path = (
            ARTICLE_TABLES_DIR
            / f"bootstrap_{file_label}.csv"
        )

        tex_path = (
            ARTICLE_TABLES_DIR
            / f"bootstrap_{file_label}.tex"
        )

        table.to_csv(
            csv_path,
            index=False,
        )

        latex_code = create_latex_table(
            article_table=table,
            comparison_name=comparison_name,
        )

        tex_path.write_text(
            latex_code,
            encoding="utf-8",
        )

        print(
            f"\nComparison: {comparison_name}"
        )
        print(f"CSV:   {csv_path.resolve()}")
        print(f"LaTeX: {tex_path.resolve()}")
        print(table.to_string(index=False))



results_original = pd.read_csv(INPUT_FILE)

results_corrected = prepare_bootstrap_results(
    results_original
)

save_article_tables(
    results_corrected
)

# Table SM 1 and SM2 in latex

In [ ]:
csv_file = Path("preprocessing_data_counts.csv")

df = pd.read_csv(csv_file)

station_labels = {
    "Camden - Euston Road": "Euston Road",
    "Camden Kerbside": "Camden Kerbside",
    "London Marylebone Road": "Marylebone Road",
    "Wandsworth - Putney High Street": "Putney High Street",
    "Westminster - Oxford Street": "Oxford Street"
}

df["station"] = df["station"].map(station_labels).fillna(df["station"])
station_order = ["Euston Road", "Camden Kerbside", "Marylebone Road", "Putney High Street", "Oxford Street"]
pollutant_order = ["NO2", "O3", "PM10", "PM25"]

df["_station_order"] = pd.Categorical(df["station"], categories=station_order, ordered=True)
df["_pollutant_order"] = pd.Categorical(df["pollutant"], categories=pollutant_order, ordered=True)

df = df.sort_values(["_station_order", "_pollutant_order"]).drop(columns=["_station_order", "_pollutant_order"]).reset_index(drop=True)


table1_cols = [
    "station",
    "pollutant",
    "n_pollution_raw",
    "n_traffic_hourly",
    "n_after_traffic_pollution_merge",
    "n_openweather_raw",
    "n_bkg0_raw",
    "n_bkg1_raw",
    "n_trf0_raw",
    "n_trf1_raw",
    "n_after_all_sources_merge"
]

table1 = df[table1_cols].copy()

table1 = table1.rename(columns={
    "station": "Station",
    "pollutant": "Pollutant",
    "n_pollution_raw": "Pollution",
    "n_traffic_hourly": "Traffic",
    "n_after_traffic_pollution_merge": "Pollution + traffic",
    "n_openweather_raw": "Weather",
    "n_bkg0_raw": "Bkg0",
    "n_bkg1_raw": "Bkg1",
    "n_trf0_raw": "Trf0",
    "n_trf1_raw": "Trf1",
    "n_after_all_sources_merge": "Final merge"
})

tabular1 = table1.to_latex(index=False, escape=False, na_rep="--", column_format="llrrrrrrrrr")

latex1 = rf"""
\begin{{table}}[H]
\centering
\caption{{Number of observations available from each data source and after merging the different data sources.}}
\label{{tab:preprocessing_data_availability}}
\resizebox{{\textwidth}}{{!}}{{%
{tabular1}
}}
\begin{{minipage}}{{\textwidth}}
\footnotesize
\textit{{Note:}} Pollution and traffic indicate the number of observations available from the target monitoring station and hourly aggregated traffic data, respectively. Bkg0 and Bkg1 denote the two nearest background stations, while Trf0 and Trf1 denote the two nearest traffic stations. Final merge indicates the number of observations remaining after merging all data sources by timestamp.
\end{{minipage}}
\end{{table}}
""".strip()

Path("preprocessing_table1.tex").write_text(latex1, encoding="utf-8")

table2_cols = [
    "station",
    "pollutant",
    "n_after_cleaning",
    "n_missing_target",
    "n_after_target_removal",
    "n_missing_predictor_values",
    "n_rows_with_missing_predictors",
    "n_interpolated_predictor_values",
    "Rows with missing predictor values (%)",
    "Missing predictor values interpolated (%)"
]

table2 = df[table2_cols].copy()

table2 = table2.rename(columns={
    "station": "Station",
    "pollutant": "Pollutant",
    "n_after_cleaning": "After cleaning",
    "n_missing_target": "Missing target",
    "n_after_target_removal": "After target removal",
    "n_missing_predictor_values": "Missing predictors",
    "n_rows_with_missing_predictors": "Rows with missing",
    "n_interpolated_predictor_values": "Interpolated values",
    "Rows with missing predictor values (%)": "Rows with missing (\%)",
    "Missing predictor values interpolated (%)": "Missing values interpolated (\%)"
})

table2["Rows with missing (\%)"] = table2["Rows with missing (\%)"].round(2)
table2["Missing values interpolated (\%)"] = table2["Missing values interpolated (\%)"].round(2)

tabular2 = table2.to_latex(index=False, escape=False, na_rep="--", float_format="%.2f", column_format="llrrrrrrrr")

latex2 = rf"""
\begin{{table}}[H]
\centering
\caption{{Number of observations and missing values during data preprocessing and temporal interpolation.}}
\label{{tab:preprocessing_missing_data}}
\resizebox{{\textwidth}}{{!}}{{%
{tabular2}
}}
\begin{{minipage}}{{\textwidth}}
\footnotesize
\textit{{Note:}} Missing target indicates observations removed because the target pollutant concentration was unavailable. Missing predictors indicates the total number of missing predictor values before interpolation, whereas Rows with missing indicates the number of observations containing at least one missing predictor. Interpolated values indicates the number of unique missing predictor values recovered by temporal interpolation. Percentages indicate the proportion of observations containing missing predictor values and the proportion of missing predictor values recovered through interpolation, respectively.
\end{{minipage}}
\end{{table}}
""".strip()

Path("preprocessing_table2.tex").write_text(latex2, encoding="utf-8")

print("Saved: preprocessing_table1.tex")
print("Saved: preprocessing_table2.tex")

print("\nTABLE 1:\n")
print(latex1)

print("\nTABLE 2:\n")
print(latex2)

Saved: preprocessing_table1.tex
Saved: preprocessing_table2.tex

TABLE 1:

\begin{table}[H]
\centering
\caption{Number of observations available from each data source and after merging the different data sources.}
\label{tab:preprocessing_data_availability}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llrrrrrrrrr}
\toprule
Station & Pollutant & Pollution & Traffic & Pollution + traffic & Weather & Bkg0 & Bkg1 & Trf0 & Trf1 & Final merge \\
\midrule
Euston Road & NO2 & 4152 & 4510 & 3747 & 4151 & 4151 & 4151 & 4151 & 4151 & 3747 \\
Camden Kerbside & NO2 & 4152 & 4706 & 3943 & 4152 & 4151 & 4151 & 4151 & 4151 & 3943 \\
Camden Kerbside & PM10 & 4152 & 4706 & 3943 & 4152 & 4151 & 4151 & 4151 & 4151 & 3943 \\
Camden Kerbside & PM25 & 4152 & 4706 & 3943 & 4152 & 4151 & 4151 & 4151 & 4151 & 3943 \\
Marylebone Road & NO2 & 4152 & 4707 & 3944 & 4152 & 4151 & 4151 & 4151 & 4151 & 3944 \\
Marylebone Road & O3 & 4152 & 4707 & 3944 & 4152 & 4151 & 4151 & 4151 & 4151 & 3944 \\
Marylebone Road & PM10 

In [4]:
df

,station,pollutant,n_pollution_raw,n_traffic_hourly,n_after_traffic_pollution_merge,n_openweather_raw,n_bkg0_raw,n_bkg1_raw,n_trf0_raw,n_trf1_raw,n_after_all_sources_merge,n_after_cleanning,n_missing_target,n_after_target_removal,n_missing_predictor_values,n_rows_with_missing_predictors,n_interpolated_predictor_values,Rows with missing predictor values (%),Missing predictor values interpolated (%)
0,Euston Road,NO2,4152,4510,3747,4151,4151,4151,4151,4151,3747,3747,515,3232,424,264,236,8.168317,55.660377
1,Camden Kerbside,NO2,4152,4706,3943,4152,4151,4151,4151,4151,3943,3943,31,3912,866,517,443,13.215746,51.154734
2,Camden Kerbside,PM10,4152,4706,3943,4152,4151,4151,4151,4151,3943,3943,103,3840,871,503,449,13.098958,51.549943
3,Camden Kerbside,PM25,4152,4706,3943,4152,4151,4151,4151,4151,3943,3943,74,3869,880,507,464,13.104161,52.727273
4,Marylebone Road,NO2,4152,4707,3944,4152,4151,4151,4151,4151,3944,3944,53,3891,612,336,304,8.635312,49.673203
5,Marylebone Road,O3,4152,4707,3944,4152,4151,4151,4151,4151,3944,3944,334,3610,560,316,293,8.753463,52.321429
6,Marylebone Road,PM10,4152,4707,3944,4152,4151,4151,4151,4151,3944,3944,243,3701,570,311,271,8.403134,47.543860
7,Marylebone Road,PM25,4152,4707,3944,4152,4151,4151,4151,4151,3944,3944,234,3710,542,295,274,7.951482,50.553506
8,Putney High Street,NO2,4152,4510,3747,4152,3747,4151,4151,4151,3747,3747,10,3737,1151,718,456,19.213273,39.617724
9,Putney High Street,PM10,4152,4510,3747,4152,3747,4151,4151,4151,3747,3747,28,3719,1139,712,443,19.144931,38.893766
